In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl
!pip install transformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-pjt6_qxp/unsloth_fb62fafa11744ec2a9718a484bd501bd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-pjt6_qxp/unsloth_fb62fafa11744ec2a9718a484bd501bd


In [ ]:
import torch
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

max_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length=max_length,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceTB/Countdown-Task-GOLD",
    "verified_Qwen2.5-7B-Instruct",
    split="train[:10000]"
)

def format_prompts(batch):
    prompts_list = batch["messages"]
    texts = []

    for messages in prompts_list:
        if len(messages) >= 3:
            user_text = messages[1]['content']
            assistant_text = messages[2]['content']
            text = f"<start_of_turn>user\n{user_text}<end_of_turn>\n<start_of_turn>model\n{assistant_text}<end_of_turn>"
        else:
            text = ""
        texts.append(text)

    return {"text": texts}


dataset = dataset.map(format_prompts, batched = True)

print(dataset[0]["text"])

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=2048,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset.column_names,
)

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported

sft_config = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    max_steps = 1500,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    push_to_hub = False,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    train_dataset = tokenized_dataset,
    args = sft_config,
    processing_class = tokenizer,
    data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False),
)

trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

prompt = "<prompt>Используя числа [8, 8, 3, 3], получи 24</prompt>\n<think>"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

print("Начинаю генерацию... (это может занять до минуты)")

outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    repetition_penalty = 1.2,
    no_repeat_ngram_size = 3,
    temperature = 0.4,
    top_p = 0.9,
    use_cache = True,
    pad_token_id = tokenizer.eos_token_id
)

print("-" * 30)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
import pandas as pd
import re
from tqdm import tqdm
from unsloth import FastLanguageModel

test_df = pd.read_csv("test_public.csv")
FastLanguageModel.for_inference(model)
results = []

print("Start test")

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    problem = row['problem']
    prompt = f"<prompt>{problem}</prompt>\n<think>"

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1024,
        use_cache = True,
        temperature = 0.1
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens = True)

    match = re.search(r'<answer>(.*?)(?:</answer>|$)', full_text, re.DOTALL)

    if match:
        answer = match.group(1).strip()
    else:
        answer = full_text.split()[-1]

    results.append({
        "id": row['id'],
        "answer": answer
    })

submission_df = pd.DataFrame(results)
submission_df.to_csv("submission.csv", index=False)

print("\nDone")